In [31]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

## RAG pipeline - Data ingestion to Vector DB pipline


In [32]:
import os
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [33]:
## read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all pdf in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #find all pdf files recursively|**→ search through all directories/subdirectories|/→ path separator|*→ any file whose name ends with .pdf
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing:{pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            #add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"✓ Loaded {len(documents)}pages")

        except Exception as e:
            print(f" X error:{e}")

    print(f"\nTotal document loader:{len(all_documents)}")
    return all_documents

#process all pdfs in the data directory
all_pdf_documents = process_all_pdfs("../data")


Found 6 PDF files to process

Processing:60_Day_Interview_Challenge_Preetham.pdf
✓ Loaded 1117pages

Processing:AI_ML_Interview_Prep.pdf
✓ Loaded 46pages

Processing:Finance_RAG_Complete_Reference.pdf
✓ Loaded 38pages

Processing:Introduction to Docker.pdf
✓ Loaded 6pages

Processing:Most_Imp_AI_ML_Interview_Questions_Only.pdf
✓ Loaded 6pages

Processing:Updated_AI_Production_Interview_Prep.pdf
✓ Loaded 42pages

Total document loader:1255


In [34]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-08T06:33:39+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-08T06:33:39+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\60_Day_Interview_Challenge_Preetham.pdf', 'total_pages': 1117, 'page': 0, 'page_label': '1', 'source_file': '60_Day_Interview_Challenge_Preetham.pdf', 'file_type': 'pdf'}, page_content='60-Day Interview\n Challenge\nComplete Preparation Guide — Fresher Track (Aggressive Mode)\nPython Fullstack\nAI / ML Engineer\nData Engineer\nData Scientist\nSoftware Tester\nSDE\n60\nDays\n600+\nInterview Qs\n600+\nPython Coding\n600+\nSQL Problems\n180+\nDSA Problems\nPrepared for: Preetham | MCA Graduate — VVIET Mysuru | 2026\nTarget: Bengaluru Companies — AI/ML · Python · Data Engineering · Testing\nEach day: 10 Interview Questions · 10 Python Coding · 10 SQL · 3 DSA Problems\n C

## Text Splitting get into chunks

In [35]:
def split_documents(documents, chunk_size=1024, chunk_overlap=128):
    """split document into small chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # show examples of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"content: {split_docs[0].page_content[:2000]}...")
        print(f"metadata: {split_docs[0].metadata}")
    
    return split_docs

In [36]:
chunks = split_documents(all_pdf_documents)
chunks

Split 1255 documents into 2542 chunks

Example chunk:
content: 60-Day Interview
 Challenge
Complete Preparation Guide — Fresher Track (Aggressive Mode)
Python Fullstack
AI / ML Engineer
Data Engineer
Data Scientist
Software Tester
SDE
60
Days
600+
Interview Qs
600+
Python Coding
600+
SQL Problems
180+
DSA Problems
Prepared for: Preetham | MCA Graduate — VVIET Mysuru | 2026
Target: Bengaluru Companies — AI/ML · Python · Data Engineering · Testing
Each day: 10 Interview Questions · 10 Python Coding · 10 SQL · 3 DSA Problems
 Complete previous day to unlock next day. Day-locked progression system....
metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-08T06:33:39+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-08T06:33:39+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\60_Day_Interview_Challenge_Preetham.pdf', 'total_pages': 1117, 'page': 0, 'page

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-08T06:33:39+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-08T06:33:39+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\60_Day_Interview_Challenge_Preetham.pdf', 'total_pages': 1117, 'page': 0, 'page_label': '1', 'source_file': '60_Day_Interview_Challenge_Preetham.pdf', 'file_type': 'pdf'}, page_content='60-Day Interview\n Challenge\nComplete Preparation Guide — Fresher Track (Aggressive Mode)\nPython Fullstack\nAI / ML Engineer\nData Engineer\nData Scientist\nSoftware Tester\nSDE\n60\nDays\n600+\nInterview Qs\n600+\nPython Coding\n600+\nSQL Problems\n180+\nDSA Problems\nPrepared for: Preetham | MCA Graduate — VVIET Mysuru | 2026\nTarget: Bengaluru Companies — AI/ML · Python · Data Engineering · Testing\nEach day: 10 Interview Questions · 10 Python Coding · 10 SQL · 3 DSA Problems\n C

## Embedding abd Vector DB

In [37]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [38]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


C:\Users\preet\AppData\Local\Temp\ipykernel_32928\540119965.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VectorStore


In [39]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 7626


In [40]:
chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-08T06:33:39+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-08T06:33:39+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\60_Day_Interview_Challenge_Preetham.pdf', 'total_pages': 1117, 'page': 0, 'page_label': '1', 'source_file': '60_Day_Interview_Challenge_Preetham.pdf', 'file_type': 'pdf'}, page_content='60-Day Interview\n Challenge\nComplete Preparation Guide — Fresher Track (Aggressive Mode)\nPython Fullstack\nAI / ML Engineer\nData Engineer\nData Scientist\nSoftware Tester\nSDE\n60\nDays\n600+\nInterview Qs\n600+\nPython Coding\n600+\nSQL Problems\n180+\nDSA Problems\nPrepared for: Preetham | MCA Graduate — VVIET Mysuru | 2026\nTarget: Bengaluru Companies — AI/ML · Python · Data Engineering · Testing\nEach day: 10 Interview Questions · 10 Python Coding · 10 SQL · 3 DSA Problems\n C

In [41]:
#convert the text to embeddings
texts = [doc.page_content for doc in chunks]

In [42]:
#genetatr the embeddings
embeddings = embedding_manager.generate_embeddings(texts)

#store in the vector store
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 2542 texts...


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Generated embeddings with shape: (2542, 384)
Adding 2542 documents to vector store...
Successfully added 2542 documents to vector store
Total documents in collection: 10168


# Retrive pipeline form vector store


In [43]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [44]:
rag_retriever


In [45]:
rag_retriever.retrieve("what is docker")

Retrieving documents for query: 'what is docker'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e90c691f_2405',
  'content': 'What is Docker?\nDocker is a platform designed to help developers build, share, and run container applications.\nWhy do we need dockers?\nConsistency Across Environments\nProblem: Applications often behave differently in development, testing, and production \nenvironments due to variations in configurations, dependencies, and infrastructure.\n•\nSolution: Docker containers encapsulate all the necessary components, ensuring the \napplication runs consistently across all environments.\n•\nIsolation\nProblem: Running multiple applications on the same host can lead to conflicts, such as \ndependency clashes or resource contention.\n•\nSolution: Docker provides isolated environments for each application, preventing \ninterference and ensuring stable performance.\n•\nScalability\nProblem: Scaling applications to handle increased load can be challenging, requiring \nmanual intervention and configuration.\n•\nSolution: Docker makes it easy to scale ap

In [46]:
rag_retriever.retrieve("What is RAG?")

Retrieving documents for query: 'What is RAG?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_2d7d893b_2347',
  'content': 'address that standard RAG always generates, even from poor context.\nQ6. What is RAFT (Retrieval-Augmented Fine-Tuning)?\nCombines RAG and fine-tuning: builds a synthetic dataset of (question, relevant docs, distractor docs, reasoning) \nfrom your corpus and fine-tunes the LLM on it, teaching it to ignore distractors and reason in domain-specific \nlanguage — useful where a general LLM misreads specialised terminology.\nPreetham  |  github.com/preetham789/finance-rag16',
  'metadata': {'source': '..\\data\\pdf\\Finance_RAG_Complete_Reference.pdf',
   'author': 'Un-named',
   'doc_index': 2347,
   'page_label': '16',
   'file_type': 'pdf',
   'producer': 'LibreOffice 24.2',
   'creator': 'Writer',
   'page': 15,
   'source_file': 'Finance_RAG_Complete_Reference.pdf',
   'creationdate': '2026-08-09T12:39:59+00:00',
   'total_pages': 38,
   'content_length': 468},
  'similarity_score': 0.1399749517440796,
  'distance': 0.8600250482559204,
  'rank

In [47]:
rag_retriever.retrieve("What is machine learning?")

Retrieving documents for query: 'What is machine learning?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_ff9ad437_1642',
  'content': 'Day 46 — Machine Learning Foundations — Advanced\nML Core Concepts · DSA: Graphs\nTopics: ML · Data Science\nSection A — Interview Questions\nQ1. Explain the key concepts and internals of this topic with a real production example.\n● HARD\nAnswer:\nIn the context of Machine Learning Foundations: This is a depth question testing your understanding of\ncore internals, trade-offs, and production readiness. A strong answer covers: (1) fundamental concepts\nwith precise terminology, (2) at least one concrete real-world example or system, (3) known pitfalls and\nhow to avoid them, (4) scalability or performance considerations at 10M+ users, and (5) tooling and\nobservability. Aim for 2–3 minutes of structured speaking with a STAR-format story for behavioural\nvariants.\nCode:\n # Day 46 — Machine Learning Foundations\n# Sample implementation pattern\ndef solution_46(data):\n"""\nProduction pattern for Machine Learning Foundations.\nTime: O(n log n) 

# RAG Pipeline - VectorDB To LLM output generation

In [48]:
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

print(os.getenv("GROQ_API_KEY"))

In [49]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [50]:
class GroqLLM:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile", api_key: str = None):
        """Initialize the Groq language model."""
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError(
                "Groq API key is required. Set GROQ_API_KEY or pass api_key."
            )

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=512,
        )
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """Generate a response using the retrieved context."""
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context does not contain enough information to answer the question, say so.""",
        )
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            response = self.llm.invoke([HumanMessage(content=formatted_prompt)])
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """Generate a response using a simple context-based prompt."""
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            response = self.llm.invoke([HumanMessage(content=simple_prompt)])
            return response.content
        except Exception as e:
            return f"Error generating response: {str(e)}"

In [51]:
try:
    groq_llm = GroqLLM()
    print("Groq LLM initialized successfully.")
except Exception as e:
    print(f"Error initializing Groq LLM: {e}")
    print("Please ensure that the GROQ_API_KEY environment variable is set correctly.")
    groq_llm = None  # Set to None if initialization fails

Initialized Groq LLM with model: llama-3.3-70b-versatile
Groq LLM initialized successfully.


In [52]:
rag_retriever.retrieve("what is docker")

Retrieving documents for query: 'what is docker'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_e90c691f_2405',
  'content': 'What is Docker?\nDocker is a platform designed to help developers build, share, and run container applications.\nWhy do we need dockers?\nConsistency Across Environments\nProblem: Applications often behave differently in development, testing, and production \nenvironments due to variations in configurations, dependencies, and infrastructure.\n•\nSolution: Docker containers encapsulate all the necessary components, ensuring the \napplication runs consistently across all environments.\n•\nIsolation\nProblem: Running multiple applications on the same host can lead to conflicts, such as \ndependency clashes or resource contention.\n•\nSolution: Docker provides isolated environments for each application, preventing \ninterference and ensuring stable performance.\n•\nScalability\nProblem: Scaling applications to handle increased load can be challenging, requiring \nmanual intervention and configuration.\n•\nSolution: Docker makes it easy to scale ap

## Integration Vectordb Context pipeline With LLM output

In [53]:
## simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

## Initialize Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key=groq_api_key, model_name="qwen/qwen3.6-27b", temperature=0.1, max_tokens=512)

# 2. simple RAG function:retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    # retrieve relevant context
    result = retriever.retrieve(query,top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in result])if result else "No relevant context found."
    if not context:
        return "No relevant context found to answer the question."
    # generate response using Groq LLM
    prompt = f"""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.
        Context: {context}  
        question: {query}
        answer:"""

    response = llm.invoke([prompt.format(context=context, question=query)])
    return response.content


In [54]:
answer = rag_simple("What is Docker?", rag_retriever, llm)
print(f"Answer: {answer}")

Retrieving documents for query: 'What is Docker?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What is Docker?"
   - **Context Provided:** Repeated text explaining what Docker is and why it's needed (Consistency, Isolation, Scalability).
   - **Key Information in Context:** "Docker is a platform designed to help developers build, share, and run container applications."

2.  **Identify Core Answer:**
   - The context explicitly states: "Docker is a platform designed to help developers build, share, and run container applications."
   - I need to answer concisely and accurately based *only* on the provided context.

3.  **Formulate Response:**
   - Directly quote or paraphrase the definition from the context.
   - Keep it concise as requested.
   - Draft: Docker is a platform designed to help developers build, share, and run container applications.

4.  **Check against Constraints:**
   - Accurate? Yes, match

## enhanced rag pipline feature

In [55]:
## ehnanced RAG pipeline feature

def rag_advanced(query,retriever,llm,top_k = 5,min_score = 0.2,return_context = False):
    """ 
    RAG pipeline with extra features:
    - Returns answer,sources,confidence score and optional full context
    """

    results = retriever.retrieve(query,top_k=top_k,score_threshold=min_score)
    if not results:
        return {
            "answer": "No relevant context found to answer the question.",
            "sources": [],
            "confidence_score": 0.0,
            "context": "" if return_context else None
        }
    # prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #generate answer

    prompt = f"""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.\nContext: {context}\nQuestion: {query}\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, question=query)])

    output = {
        "answer": response.content,
        "sources": sources,
        "confidence_score": confidence,
        "context": context if return_context else None
    }
    if return_context:
        output["context"] = context
        

    return output


In [56]:
result = rag_advanced("What is Docker?", rag_retriever, llm, top_k=3, min_score=0.2, return_context=True)
print(f"Answer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Confidence Score: {result['confidence_score']}")
print(f"Context: {result['context'][:500]}...")  # Print first 500 characters of context

Retrieving documents for query: 'What is Docker?'
Top K: 3, Score threshold: 0.2
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: 
<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** Provides a definition of Docker and explains why it's needed (Consistency, Isolation, Scalability). The context is repeated three times, but the content is identical.
   - **Question:** "What is Docker?"
   - **Task:** Answer the question accurately and concisely based *only* on the provided context.

2.  **Extract Relevant Information from Context:**
   - The context explicitly states: "Docker is a platform designed to help developers build, share, and run container applications."

3.  **Formulate Answer:**
   - Keep it direct and aligned with the context.
   - Draft: Docker is a platform designed to help developers build, share, and run container applications.
   - Check against constraints: Accurate? Yes. Concise? Yes. Based on context? Yes.

4.  **Final Output Generation:**
   - Output matches the draft. No extr

In [59]:
# advanced rag with streaming,citation and confidence score,history and context management

from typing import List, Dict, Any
import time 

class AdvancedRAG:
    def __init__(self,retriever,llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # To maintain conversation history

    def query(self,question:str,top_k:int = 5,min_score:float = 0.2,stream :bool= False,summarize:bool = False) -> Dict[str,Any]:
        #retrieve relevant documents
        results = self.retriever.retrieve(question,top_k=top_k,score_threshold = min_score)
        if not results:
            answer = "No relevant context found"
            sources = []
            context = ""

        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            }for doc in results]
            #streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("streaming answer:")
                for i in range(0,len(prompt),80):
                    print(prompt[i:i+80],end = '',flush = True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context = context,question = question)])
            answer = response.content

        #add citation to answer
        citations =[f"[{i+1}]{src['source']} (page{src['page']})"for i,src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n"+"\n".join(citations) if citations else answer

        #optional summerize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAG(rag_retriever, llm)
result = adv_rag.query("what is docker", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is docker'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
streaming answer:
Use the following context to answer the question concisely.
Context:
What is Docker?
Docker is a platform designed to help developers build, share, and run container applications.
Why do we need dockers?
Consistency Across Environments
Problem: Applications often behave differently in development, testing, and production 
environments due to variations in configurations, dependencies, and infrastructure.
•
Solution: Docker containers encapsulate all the necessary components, ensuring the 
application runs consistently across all environments.
•
Isolation
Problem: Running multiple applications on the same host can lead to conflicts, such as 
dependency clashes or resource contention.
•
Solution: Docker provides isolated environments for each application, preventing 
interference and ensuring stable performance.
•
Scalability
Problem: Scaling applications to handle increased load can be ch